# Governance & Operational Segment Telemetry Audit - Project 1
> **Production Alignment Note:** Loads the registered `Production` version of `fraud_detection_xgb` via `MlflowClient` and evaluates the full held-out `data/processed/test.parquet` (`n = 56,746`, `95` frauds) at the served `0.50` threshold using 95% Wilson recall CIs and 95% Katz segment flag-rate ratio CIs (`src/governance/audit.py`). Transaction amount tier is an operational segment, not a legal demographic protected class.

In [ ]:
import sys
from pathlib import Path
import mlflow
import mlflow.xgboost
import pandas as pd
from mlflow.tracking import MlflowClient

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.governance.audit import (
    SERVED_DECISION_THRESHOLD,
    _katz_95_ci,
    _segment_metrics,
)
from src.ingestion.utils import CANONICAL_FEATURE_COLS

DB_PATH = PROJECT_ROOT.parent / "mlflow_db" / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")
client = MlflowClient()
versions = client.search_model_versions("name='fraud_detection_xgb'")
prod_versions = [v for v in versions if v.current_stage == "Production"]
if not prod_versions:
    raise RuntimeError("No Production version found for fraud_detection_xgb")
selected_mv = max(prod_versions, key=lambda v: int(v.version))
model_uri = f"models:/fraud_detection_xgb/{selected_mv.version}"
model = mlflow.xgboost.load_model(model_uri)
print(f"Loaded Production model: {model_uri} (run_id={selected_mv.run_id})")

In [ ]:
test_df = pd.read_parquet(PROJECT_ROOT / "data/processed/test.parquet")
X_test = test_df[CANONICAL_FEATURE_COLS]
y_test = test_df["Class"].to_numpy(dtype=int)

probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= SERVED_DECISION_THRESHOLD).astype(int)
overall = _segment_metrics(y_test, preds)
q80 = float(test_df["scaled_amount"].quantile(0.80))
is_high = (test_df["scaled_amount"] >= q80).to_numpy()
high_seg = _segment_metrics(y_test[is_high], preds[is_high])
low_seg = _segment_metrics(y_test[~is_high], preds[~is_high])
di_ratio = (
    round(high_seg["flag_rate"] / low_seg["flag_rate"], 4)
    if low_seg["flag_rate"] > 0
    else 0.0
)
di_ci = _katz_95_ci(
    high_seg["flagged_count"],
    high_seg["count"],
    low_seg["flagged_count"],
    low_seg["count"],
)
print(
    "Overall Holdout Recall:",
    overall["recall"],
    "95% Wilson CI:",
    overall["recall_95_ci"],
    "Precision:",
    overall["precision"],
)
print("High-Value Segment (>=80th pct):", high_seg)
print("Low-Value Segment (<80th pct):", low_seg)
print(f"Informational Segment Flag-Rate Ratio: {di_ratio} (95% Katz CI: {di_ci})")